# Mule Strategy — neighbour count, seed overlap, feature filter

Run **after** `mule_validation.ipynb` (which caches `profiles_cache.parquet`).
This notebook promotes confirmed mules to seeds alongside Frontier/RAVO and runs
three analyses grounded in Baesens *Fraud Analytics*:

1. **Neighbour-count sweep** — how recall (mules caught) and list size (workload)
   trade off as you lengthen the per-seed list. Cut-off analysis (Ch. 4).
2. **Multi-seed overlap** — rank accounts by how many confirmed mules they resemble;
   overlap is signal, not noise.
3. **Fisher-score filter** — which features actually separate mule from non-mule
   (Ch. 2 variable selection). Tells us if the feature set needs changing.

**Leave-one-out is used throughout:** a mule is never scored against itself as a seed,
so recall numbers are honest (no train/test leakage).

*Small-n caveat:* with ~10 labels these are directional. Recall and list size are
trustworthy; precision is only a proxy (an unlabelled flagged account is not a
confirmed non-mule), so read it as workload, not accuracy.

## 1. Config

In [1]:
# Same confirmed-mule list as the validation notebook
CONFIRMED_MULES = [
    "UG6119095",   # <-- placeholder, replace with a real confirmed ACID
    "UG6119094",   # <-- placeholder
    "UG6119032",   # <-- placeholder
    "UG6052746",   # <-- placeholder   # <-- placeholder
    "UG6112062",   # <-- placeholder
    "UG6094534",
    "UG6154756",
    "UG6310606",
    # add more here, one ACID per line
]
# Existing reference accounts to keep as seeds alongside the mules
EXTRA_REFERENCES = ["UG6246002", "UG6370211"]   # Frontier, RAVO

NEIGHBOUR_KS = [10, 20, 30, 50]   # per-seed list lengths to sweep
OVERLAP_K    = 30                 # neighbour window for the overlap analysis

PROFILES_CACHE = "profiles_cache.parquet"   # produced by mule_validation.ipynb

## 2. Load cached profiles, build the seed set, precompute ranks

In [2]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
load_dotenv(dotenv_path='.env', override=False)
import aml_screening as m

try:
    profiles = pd.read_parquet(PROFILES_CACHE)
except FileNotFoundError:
    print('No cache found - rebuilding from Finacle (run mule_validation first to avoid this)')
    profiles = m.build_profiles(m.clean_transactions(m.extract_transactions_db()))
    profiles.to_parquet(PROFILES_CACHE)
profiles = profiles.reset_index(drop=True)

X = m.scale_features(profiles)
N = len(profiles)
acid_to_row = {a: i for i, a in enumerate(profiles['ACID'])}
name_of = dict(zip(profiles['ACID'], profiles['ACCT_NAME']))

in_scope = [a for a in CONFIRMED_MULES if a in acid_to_row]
seeds = [s for s in dict.fromkeys(EXTRA_REFERENCES + in_scope) if s in acid_to_row]
mule_rows = {acid_to_row[a] for a in in_scope}

print(f"Book: {N:,} accounts | mules in scope: {len(in_scope)} | total seeds: {len(seeds)}")

# per-seed rank of every account (rank 1 = the seed itself)
seed_rank = {}
for s in seeds:
    d = np.linalg.norm(X - X[acid_to_row[s]], axis=1)
    order = np.argsort(d)
    rank = np.empty(N, dtype=int)
    rank[order] = np.arange(1, N + 1)
    seed_rank[s] = rank

Book: 23,602 accounts | mules in scope: 8 | total seeds: 10


## 3. Analysis 1 — neighbour-count sweep (recall vs workload)

For each per-seed list length K: **recall** = held-out mules caught in the top-K of
at least one *other* seed; **flagged_total** = distinct accounts surfaced across all
seeds (the review workload); **confirmed_per_100** = a rough precision proxy.

In [3]:
def loo_best_rank(a):
    """Best rank of mule a across all seeds except itself (leave-one-out)."""
    im = acid_to_row[a]
    rs = [seed_rank[s][im] for s in seeds if s != a]
    return min(rs) if rs else 10**9

rows1 = []
for K in NEIGHBOUR_KS:
    caught = sum(1 for a in in_scope if 2 <= loo_best_rank(a) <= K + 1)
    flagged = set()
    for s in seeds:
        rk = seed_rank[s]
        flagged.update(np.where((rk >= 2) & (rk <= K + 1))[0].tolist())
    conf = len(flagged & mule_rows)
    rows1.append({
        'K_per_seed': K,
        'recall_mules': f"{caught}/{len(in_scope)}",
        'flagged_total': len(flagged),
        'confirmed_in_list': conf,
        'confirmed_per_100': round(100 * conf / max(len(flagged), 1), 2),
    })
sweep1 = pd.DataFrame(rows1)
sweep1

,K_per_seed,recall_mules,flagged_total,confirmed_in_list,confirmed_per_100
0,10,7/8,69,7,10.14
1,20,7/8,114,7,6.14
2,30,8/8,160,8,5.00
3,50,8/8,242,8,3.31


## 4. Analysis 2 — multi-seed overlap

How many seeds is each account a neighbour of (within top-`OVERLAP_K`)? An account
close to several confirmed mules is a stronger lead than one close to a single seed.

In [4]:
K = OVERLAP_K
overlap = np.zeros(N, dtype=int)
for s in seeds:
    rk = seed_rank[s]
    overlap[(rk >= 2) & (rk <= K + 1)] += 1

tbl = profiles[['ACID', 'ACCT_NAME', 'account_status']].copy()
tbl['seed_overlap'] = overlap
tbl['known_mule'] = tbl['ACID'].isin(in_scope)

mule_ov = overlap[list(mule_rows)]
print(f"Known mules   : median overlap {np.median(mule_ov):.0f}, max {mule_ov.max()}, "
      f">=2 seeds {(mule_ov >= 2).sum()}/{len(mule_ov)}")
print(f"Whole book    : median overlap {np.median(overlap):.0f}")
print("\nTop accounts by seed overlap (leads — high overlap, not already known):")
tbl[~tbl['known_mule']].sort_values('seed_overlap', ascending=False).head(15)

Known mules   : median overlap 2, max 3, >=2 seeds 6/8
Whole book    : median overlap 0

Top accounts by seed overlap (leads — high overlap, not already known):


,ACID,ACCT_NAME,account_status,seed_overlap,known_mule
13277,UG6162364,PRAND ENGINEERING LIMITED,Open,5,False
11545,UG6127102,PANORAMA PETROLEUM LIMITED,Open,4,False
14207,UG6180922,LULIRO IMPORT & EXPORT COMPANY LIMITED,Open,4,False
13149,UG6158629,FASBIR ENTERPRISES,Open,4,False
11852,UG6133078,ZEPHYR TRADING (U) LIMITED,Open,4,False
1862,UG5958878,ELITE HOPE UGANDA LTD,Open,4,False
5264,UG6018842,GG TECH SYSTEMS LIMITED,Closed,4,False
15513,UG6209071,TAURARO INVESTMENT LIMITED,Open,4,False
13766,UG6173878,MERCURY SHIPPING CO LIMITED,Open,4,False
12310,UG6143987,ABO LEENA MULTI ACTIVITIES CO. LTD,Open,4,False


## 5. Analysis 3 — Fisher-score feature filter

Fisher score = (mule_mean - book_mean)^2 / (mule_var + book_var). Higher = the feature
separates mules from the rest better. Ranks the current feature set; a candidate new
feature is only worth adding if it scores near the top here.

In [5]:
y = profiles['ACID'].isin(in_scope).values
rows3 = []
for f in m.BEHAVIOUR_COLS:
    v = profiles[f]
    v1, v0 = v[y].dropna(), v[~y].dropna()
    den = v1.var() + v0.var()
    fisher = float((v1.mean() - v0.mean())**2 / den) if den and den > 0 else np.nan
    rows3.append({'feature': f, 'mule_mean': round(v1.mean(), 3),
                  'book_mean': round(v0.mean(), 3), 'fisher_score': round(fisher, 4)})
fisher_tbl = pd.DataFrame(rows3).sort_values('fisher_score', ascending=False)
fisher_tbl

,feature,mule_mean,book_mean,fisher_score
8,log_turnover_per_day,10.936,4.657,4.6687
3,top3_credit_share,0.201,0.792,3.1435
4,top3_debit_share,0.171,0.751,2.9252
6,frac_credits_followed_24h,0.883,0.337,2.4005
2,largest_debit_share,0.070,0.580,1.9057
1,largest_credit_share,0.085,0.554,1.8971
5,median_hours_to_next_debit,2.635,476.686,0.1083
7,account_age_days,325.625,297.275,0.0082
0,flow_through_ratio,0.998,5.985,0.0004


## 6. PASTE THIS BACK — copy the whole output of this cell

In [6]:
print("=" * 62)
print("MULE STRATEGY SUMMARY")
print("=" * 62)
print(f"Accounts {N:,} | mules in scope {len(in_scope)} | seeds {len(seeds)}")
print()
print("A1  neighbour-count sweep (leave-one-out recall):")
print(f"    {'K':>4} {'recall':>8} {'flagged':>9} {'conf/100':>9}")
for r in rows1:
    print(f"    {r['K_per_seed']:>4} {r['recall_mules']:>8} "
          f"{r['flagged_total']:>9} {r['confirmed_per_100']:>9}")
print()
print(f"A2  seed overlap (K={OVERLAP_K}): known-mule median "
      f"{np.median(mule_ov):.0f} (max {mule_ov.max()}), book median {np.median(overlap):.0f}")
lead = tbl[~tbl['known_mule']].sort_values('seed_overlap', ascending=False).head(5)
print("    top non-mule leads by overlap:")
for _, r in lead.iterrows():
    print(f"      overlap {r['seed_overlap']:>2}  {r['ACCT_NAME'][:34]:<34} [{r['account_status']}]")
print()
print("A3  Fisher scores (feature separation, high = better):")
for _, r in fisher_tbl.iterrows():
    print(f"    {r['feature']:<28} {r['fisher_score']:>8}  "
          f"(mule {r['mule_mean']} vs book {r['book_mean']})")
print("=" * 62)

MULE STRATEGY SUMMARY
Accounts 23,602 | mules in scope 8 | seeds 10

A1  neighbour-count sweep (leave-one-out recall):
       K   recall   flagged  conf/100
      10      7/8        69     10.14
      20      7/8       114      6.14
      30      8/8       160       5.0
      50      8/8       242      3.31

A2  seed overlap (K=30): known-mule median 2 (max 3), book median 0
    top non-mule leads by overlap:
      overlap  5  PRAND ENGINEERING LIMITED          [Open]
      overlap  4  PANORAMA PETROLEUM LIMITED         [Open]
      overlap  4  LULIRO IMPORT & EXPORT COMPANY LIM [Open]
      overlap  4  FASBIR ENTERPRISES                 [Open]
      overlap  4  ZEPHYR TRADING (U) LIMITED         [Open]

A3  Fisher scores (feature separation, high = better):
    log_turnover_per_day           4.6687  (mule 10.936 vs book 4.657)
    top3_credit_share              3.1435  (mule 0.201 vs book 0.792)
    top3_debit_share               2.9252  (mule 0.171 vs book 0.751)
    frac_credits_fol

In [7]:
profiles

,ACID,total_credit,total_debit,credit_count,debit_count,max_credit,max_debit,median_hours_to_next_debit,credits_followed_24h,top3_credit_share,...,LAST_TRAN_DATE,flow_through_ratio,largest_credit_share,largest_debit_share,frac_credits_followed_24h,account_age_days,account_status,turnover,turnover_per_day,log_turnover_per_day
0,UG5922931,2149.45,2051.45,5.0,6.0,2048.00,1100.00,5.354722,4.0,0.994603,...,2026-01-31,0.954407,0.952802,0.536206,0.800000,760,Open,4200.90,5.527500,1.876024
1,UG5923174,3104.53,3016.14,6.0,4.0,3030.00,1850.00,4.822083,4.0,0.997330,...,2026-01-03,0.971529,0.975993,0.613367,0.666667,732,Open,6120.67,8.361571,2.236613
2,UG5923233,157008.65,119027.00,102.0,13.0,3600.00,30000.00,1681.735000,5.0,0.060458,...,2026-08-01,0.758092,0.022929,0.252044,0.049020,942,Open,276035.65,293.031476,5.683687
3,UG5923277,347780.00,344845.35,39.0,220.0,92000.00,89500.00,44.706389,13.0,0.437058,...,2026-07-31,0.991562,0.264535,0.259537,0.333333,940,Open,692625.35,736.835479,6.603721
4,UG5923418,5085.42,5064.28,6.0,13.0,2160.00,1800.00,145.068750,2.0,0.790487,...,2024-10-12,0.995843,0.424744,0.355431,0.333333,283,Open,10149.70,35.864664,3.607253
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23597,UG6419673,46.76,20.79,2.0,1.0,25.97,20.79,4.888472,2.0,1.000000,...,2026-08-01,0.444611,0.555389,1.000000,1.000000,0,Open,67.55,NaN,NaN
23598,UG6419533,110.24,107.00,1.0,2.0,110.24,100.00,0.272778,1.0,1.000000,...,2026-08-01,0.970610,1.000000,0.934579,1.000000,0,Open,217.24,NaN,NaN
23599,UG6419208,103.90,0.00,1.0,0.0,103.90,NaN,NaN,0.0,1.000000,...,2026-07-31,0.000000,1.000000,NaN,0.000000,0,Open,103.90,NaN,NaN
23600,UG6419706,48.05,0.00,1.0,0.0,48.05,NaN,NaN,0.0,1.000000,...,2026-08-01,0.000000,1.000000,NaN,0.000000,0,Open,48.05,NaN,NaN
